In [ ]:
import spacy
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from setfit import SetFitModel, SetFitTrainer
from setfit import sample_dataset
from sentence_transformers import SentenceTransformer
from pathlib import Path

In [ ]:
def extract_candidates(doc):
    return [chunk.text for chunk in doc.noun_chunks]

def get_data(reviews_text):
    nlp = spacy.load("en_core_web_lg", disable=["ner", "textcat"])
    results = []
    texts = []

    with open(reviews_text, 'r', encoding="utf-8") as f:
        for line in f:
            review = json.loads(line)
            results.append(review)
            texts.append(review['review_text'])
    
    candidates_list = []
    for doc in nlp.pipe(texts, batch_size=64, n_process=-1):
        candidates_list.append(extract_candidates(doc))

    for review, candidates in zip(results, candidates_list):
        review['candidates'] = candidates

    return pd.DataFrame(results)

data = Path("data/final_labelled_reviews.json")
reviews_df = get_data(data)
print(reviews_df.head())


In [ ]:
train_df, test_df = train_test_split(reviews_df, test_size=0.2, random_state=42, stratify=reviews_df['label'])

train_texts = train_df['review_text'].tolist()
train_labels = train_df['label'].tolist()
test_texts = test_df['review_text'].tolist()
test_labels = test_df['label'].tolist()


In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-mpnet-base-v2")


In [ ]:
model = SetFitModel.from_sentence_transformer(embedding_model)

trainer = SetFitTrainer(
    model=model,
    train_texts=train_texts,
    train_labels=train_labels,
    eval_texts=test_texts,
    eval_labels=test_labels,
    metric="accuracy",
    batch_size=16,
    num_iterations=20,
)

trainer.train()


In [ ]:
model.save_pretrained("setfit-electronics-aspect")


In [ ]:
model = SetFitModel.from_pretrained("setfit-electronics-aspect")
new_review = "The battery lasts long but the screen is dull."
nlp = spacy.load("en_core_web_lg", disable=["ner", "textcat"])
candidates = extract_candidates(nlp(new_review))
preds = model.predict(candidates)

for cand, pred in zip(candidates, preds):
    print("text: ", new_review, cand, "->", pred) 
